# Multimodal MOA competition: submission template

This notebook demonstrates the minimum workflow for preparing a leaderboard submission. The example uses the L1000 expression modality and a scikit-learn classifier. Students may replace it with a stronger model or combine expression, image, and SMILES features.

The important submission artifacts are:

1. `predictions.csv` with one prediction for every row of `test.csv`;
2. one or more fitted model files, such as `.joblib`, `.pkl`, or a framework checkpoint.

The leaderboard stores uploaded models for inspection and size accounting. It does **not** import, deserialize, or execute them.

## Submission rules

- Use the provided training and test split.
- Keep the rows in `test.csv` order.
- Submit exactly one prediction per `sample_id`.
- The prediction file must contain `sample_id` and `predicted_class`.
- `predicted_class` must be one of the classes observed in the training data.
- Do not use hidden targets or private metadata.
- Keep all doses of one compound in the same train/validation partition.
- You have three accepted leaderboard submissions.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Run this notebook with the working directory set to student_release.
DATA_DIR = Path('.')
ARTIFACT_DIR = Path('submission_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42

In [2]:
train_meta = pd.read_csv(DATA_DIR / 'train.csv')
test_meta = pd.read_csv(DATA_DIR / 'test.csv')
expression_train = pd.read_parquet(DATA_DIR / 'expression_train.parquet')
expression_test = pd.read_parquet(DATA_DIR / 'expression_test.parquet')

print('train metadata:', train_meta.shape)
print('test metadata:', test_meta.shape)
print('train expression:', expression_train.shape)
print('test expression:', expression_test.shape)
print('target counts:\n', train_meta['competition_class'].value_counts())

assert train_meta['sample_id'].is_unique
assert test_meta['sample_id'].is_unique
assert 'competition_class' in train_meta.columns
assert 'competition_class' not in test_meta.columns

train metadata: (1272, 8)
test metadata: (324, 7)
train expression: (1272, 980)
test expression: (324, 980)
target counts:
 competition_class
Nuclear receptor signaling                 288
MAPK / RTK signaling                       204
DNA damage / replication                   180
DRD2-targeting GPCR                        174
PI3K / AKT / mTOR signaling                114
Mitotic / cell-cycle disruption            114
Epigenetic / transcriptional regulation    108
Proteostasis / protein synthesis            90
Name: count, dtype: int64


## Build an expression feature matrix

The expression tables are keyed by `sample_id`. We select the same expression columns for train and test and then reorder rows using the metadata tables. This keeps the target and features aligned.

The identifiers are not predictive features, so they are removed from the matrix.

In [3]:
ID_COLUMNS = ['sample_id', 'expression_id']
feature_columns = [
    column for column in expression_train.columns
    if column not in ID_COLUMNS
]

train_expression_by_id = expression_train.set_index('sample_id')
test_expression_by_id = expression_test.set_index('sample_id')

X = train_expression_by_id.loc[train_meta['sample_id'], feature_columns]
X_test = test_expression_by_id.loc[test_meta['sample_id'], feature_columns]
y = train_meta['competition_class'].astype(str)
groups = train_meta['compound_id']

assert X.shape[0] == train_meta.shape[0]
assert X_test.shape[0] == test_meta.shape[0]
assert list(X_test.index) == test_meta['sample_id'].tolist()
print('features:', X.shape[1])
print('missing values in train:', int(X.isna().sum().sum()))

features: 978
missing values in train: 5868


## Make a compound-aware validation split

There are several dose measurements for each compound. A random row split would put different doses of the same compound into both partitions and make validation too optimistic. `StratifiedGroupKFold` keeps compounds together while approximately preserving class proportions.

The final model below will be refit on all public training data after this validation check.

In [4]:
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
train_indices, validation_indices = next(splitter.split(X, y, groups=groups))

X_fit = X.iloc[train_indices]
X_validation = X.iloc[validation_indices]
y_fit = y.iloc[train_indices]
y_validation = y.iloc[validation_indices]

print('fit rows:', len(train_indices))
print('validation rows:', len(validation_indices))
print('overlapping compounds:', len(set(groups.iloc[train_indices]) & set(groups.iloc[validation_indices])))
assert set(groups.iloc[train_indices]).isdisjoint(set(groups.iloc[validation_indices]))

fit rows: 1014
validation rows: 258
overlapping compounds: 0


## Train a baseline model

A pipeline is useful for submissions because it stores preprocessing and the estimator together. The median imputer and scaler are fitted only on the training partition, avoiding validation leakage.

In [5]:
model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)),
])

model.fit(X_fit, y_fit)
validation_predictions = model.predict(X_validation)
validation_f1 = f1_score(y_validation, validation_predictions, average='macro')

print(f'validation macro-F1: {validation_f1:.4f}')
print(classification_report(y_validation, validation_predictions, zero_division=0))

C:\Users\user\Desktop\kse\practicals\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


validation macro-F1: 0.4322
                                         precision    recall  f1-score   support

               DNA damage / replication       0.23      0.28      0.25        36
                    DRD2-targeting GPCR       0.31      0.47      0.38        36
Epigenetic / transcriptional regulation       0.38      0.21      0.27        24
                   MAPK / RTK signaling       0.61      0.64      0.62        36
        Mitotic / cell-cycle disruption       0.56      0.42      0.48        24
             Nuclear receptor signaling       0.57      0.62      0.59        60
            PI3K / AKT / mTOR signaling       0.57      0.50      0.53        24
       Proteostasis / protein synthesis       0.67      0.22      0.33        18

                               accuracy                           0.46       258
                              macro avg       0.49      0.42      0.43       258
                           weighted avg       0.48      0.46      0.45       25

## Refit on all public training data and create predictions

After model selection, refit the chosen pipeline using all rows in `train.csv`. The hidden test labels are not needed to create the submission.

In [6]:
final_model = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(max_iter=1000, n_jobs=-1, random_state=RANDOM_STATE)),
])
final_model.fit(X, y)
test_predictions = final_model.predict(X_test).astype(str)

predictions = pd.DataFrame({
    'sample_id': test_meta['sample_id'],
    'predicted_class': test_predictions,
})

assert len(predictions) == len(test_meta)
assert predictions['sample_id'].tolist() == test_meta['sample_id'].tolist()
assert predictions['sample_id'].is_unique
assert set(predictions['predicted_class']).issubset(set(y.unique()))

prediction_path = ARTIFACT_DIR / 'predictions.csv'
predictions.to_csv(prediction_path, index=False)
print(prediction_path)
display(predictions.head())

C:\Users\user\Desktop\kse\practicals\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


submission_artifacts\predictions.csv


,sample_id,predicted_class
0,S000043,DRD2-targeting GPCR
1,S000044,PI3K / AKT / mTOR signaling
2,S000045,DNA damage / replication
3,S000046,Mitotic / cell-cycle disruption
4,S000047,DNA damage / replication


## Save the fitted model

Save the complete pipeline rather than only the classifier. This preserves the imputer, scaler, feature order, and classifier as one reproducible artifact.

If your solution uses several independently trained models, save each one into this same directory and upload all of them in the leaderboard's **Model files** field.

In [7]:
model_path = ARTIFACT_DIR / 'expression_logistic_pipeline.joblib'
joblib.dump(final_model, model_path, compress=3)

metadata = {
    'modality': 'L1000 expression',
    'model_type': 'scikit-learn LogisticRegression pipeline',
    'n_features': len(feature_columns),
    'feature_columns': feature_columns,
    'validation_macro_f1': float(validation_f1),
    'random_state': RANDOM_STATE,
}
with (ARTIFACT_DIR / 'training_metadata.json').open('w', encoding='utf-8') as handle:
    json.dump(metadata, handle, indent=2)

print(f'saved model: {model_path} ({model_path.stat().st_size / 1024**2:.3f} MB)')
print(f'saved predictions: {prediction_path} ({prediction_path.stat().st_size / 1024:.1f} KB)')
print('artifacts:', [path.name for path in sorted(ARTIFACT_DIR.iterdir())])

saved model: submission_artifacts\expression_logistic_pipeline.joblib (0.089 MB)
saved predictions: submission_artifacts\predictions.csv (11.2 KB)
artifacts: ['expression_logistic_pipeline.joblib', 'predictions.csv', 'training_metadata.json']


## Optional: save another model

The leaderboard accepts multiple model files. For example, if you train an image model, a SMILES model, and an expression model, save each fitted artifact separately.

```python
joblib.dump(image_model, ARTIFACT_DIR / 'image_model.joblib', compress=3)
joblib.dump(smiles_model, ARTIFACT_DIR / 'smiles_model.joblib', compress=3)
```

Only upload model files that are needed to document or reproduce your submitted predictions. The leaderboard counts the combined size of all uploaded model files.

## Upload checklist

In the leaderboard form:

1. Enter the same name for all of your attempts.
2. Select `submission_artifacts/predictions.csv` as **Prediction CSV**.
3. Select `submission_artifacts/expression_logistic_pipeline.joblib` and any additional fitted model files as **Model files**.
4. Optionally upload this notebook for instructor inspection.
5. Add a short description of the modalities, preprocessing, and models.

Before submitting, verify that the prediction CSV has exactly the required columns and that no hidden target file is present in the artifact directory.

In [8]:
# Final local verification before upload.
submission = pd.read_csv(prediction_path)
assert list(submission.columns) == ['sample_id', 'predicted_class']
assert len(submission) == len(test_meta)
assert submission['sample_id'].equals(test_meta['sample_id'])
assert submission['predicted_class'].notna().all()
print('Ready for upload:', prediction_path)
print('Model files:', [path.name for path in ARTIFACT_DIR.glob('*.joblib')])

Ready for upload: submission_artifacts\predictions.csv
Model files: ['expression_logistic_pipeline.joblib']
